# 🚢 Titanic Survival Classification
## Notebook 4 — Model Training & Evaluation

### Objective

Build and compare multiple machine-learning classification models
for predicting Titanic passenger survival.

### Models

1. Logistic Regression
2. Decision Tree
3. Random Forest
4. Gradient Boosting
5. HistGradientBoosting

### Evaluation Metrics

- Accuracy
- Precision
- Recall
- F1 Score
- ROC-AUC

### Validation Strategy

A stratified train/test split is used for final holdout evaluation,
while 5-fold Stratified Cross-Validation is used for model comparison.

The preprocessing steps are placed inside a Scikit-learn Pipeline
to prevent data leakage.


In [ ]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate
)

from sklearn.linear_model import LogisticRegression

from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

import joblib

# ============================================================
# PROJECT PATH
# ============================================================

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project root:")
print(PROJECT_ROOT)


## 1. Load Engineered Dataset


In [ ]:
# ============================================================
# LOAD ENGINEERED DATA
# ============================================================

DATA_PATH = (
    PROJECT_ROOT /
    "data" /
    "processed" /
    "titanic_engineered.csv"
)

df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")

display(df.head())


In [ ]:
# ============================================================
# TARGET AND FEATURES
# ============================================================

TARGET = "Survived"

X = df.drop(
    columns=[
        TARGET,
        "PassengerId",
        "Name",
        "Ticket",
        "Cabin"
    ],
    errors="ignore"
)

y = df[TARGET]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
display(
    y.value_counts()
    .rename_axis("Survived")
    .to_frame("Count")
)


## 2. Train / Holdout Split

The dataset is split into:

- 80% training data
- 20% holdout data

Stratification is used to preserve the proportion of survivors
and non-survivors in both subsets.


In [ ]:
# ============================================================
# TRAIN / HOLDOUT SPLIT
# ============================================================

RANDOM_STATE = 42

X_train, X_holdout, y_train, y_holdout = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training data:", X_train.shape)
print("Holdout data :", X_holdout.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nHoldout target distribution:")
print(y_holdout.value_counts(normalize=True))


## 3. Identify Feature Types

Numerical variables will use:

- Median imputation
- Standard scaling

Categorical variables will use:

- Most-frequent imputation
- One-hot encoding

The transformations are contained inside the machine-learning
pipeline so that they are fitted only on the training folds.


In [ ]:
# ============================================================
# IDENTIFY NUMERICAL AND CATEGORICAL FEATURES
# ============================================================

numeric_features = X.select_dtypes(
    include=[
        "int64",
        "int32",
        "float64",
        "float32"
    ]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=[
        "object",
        "category",
        "bool"
    ]
).columns.tolist()

print("Numerical features:")
for feature in numeric_features:
    print("  •", feature)

print("\nCategorical features:")
for feature in categorical_features:
    print("  •", feature)


In [ ]:
# ============================================================
# PREPROCESSING PIPELINES
# ============================================================

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ],
    remainder="drop"
)

print("Preprocessing pipeline created successfully.")


## 4. Define Candidate Models

Several algorithms are compared rather than assuming that one
model will automatically perform best.


In [ ]:
# ============================================================
# MACHINE LEARNING MODELS
# ============================================================

models = {

    "Logistic Regression": LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=500,
        max_depth=8,
        min_samples_leaf=3,
        max_features="sqrt",
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=300,
        learning_rate=0.03,
        max_depth=3,
        min_samples_leaf=5,
        random_state=RANDOM_STATE
    ),

    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.05,
        max_leaf_nodes=15,
        l2_regularization=1.0,
        random_state=RANDOM_STATE
    )
}

print("Models prepared:")

for model_name in models:
    print("  •", model_name)


## 5. Five-Fold Stratified Cross-Validation

Stratified cross-validation ensures that each fold approximately
preserves the original survival-class distribution.

ROC-AUC is used as the primary model-comparison metric, while
accuracy, precision, recall and F1 are also recorded.


In [ ]:
# ============================================================
# CROSS-VALIDATION
# ============================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_results = []

for model_name, model in models.items():

    print(
        f"\nRunning cross-validation: {model_name}"
    )

    pipeline = Pipeline(
        steps=[
            (
                "preprocessing",
                preprocessor
            ),
            (
                "model",
                model
            )
        ]
    )

    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False
    )

    cv_results.append({
        "Model": model_name,
        "Accuracy": scores["test_accuracy"].mean(),
        "Precision": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
        "F1": scores["test_f1"].mean(),
        "ROC-AUC": scores["test_roc_auc"].mean(),

        "Accuracy_STD": scores["test_accuracy"].std(),
        "ROC-AUC_STD": scores["test_roc_auc"].std()
    })


results_df = pd.DataFrame(
    cv_results
).sort_values(
    "ROC-AUC",
    ascending=False
).reset_index(drop=True)

display(
    results_df.round(4)
)


## 6. Visualize Model Comparison


In [ ]:
# ============================================================
# MODEL COMPARISON VISUALIZATION
# ============================================================

sns.set_theme(
    style="whitegrid",
    context="notebook"
)

plt.figure(
    figsize=(12, 7)
)

ax = sns.barplot(
    data=results_df,
    x="ROC-AUC",
    y="Model",
    hue="Model",
    palette=[
        "#2563EB",
        "#7C3AED",
        "#10B981",
        "#F59E0B",
        "#EF4444"
    ],
    legend=False
)

plt.title(
    "Model Comparison — ROC-AUC",
    fontsize=18,
    fontweight="bold",
    color="#0B1F3A"
)

plt.xlabel("Mean ROC-AUC")
plt.ylabel("")

plt.xlim(
    max(0, results_df["ROC-AUC"].min() - 0.05),
    min(1, results_df["ROC-AUC"].max() + 0.05)
)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.3f",
        padding=5
    )

plt.tight_layout()

FIGURE_DIR = (
    PROJECT_ROOT /
    "outputs" /
    "figures"
)

FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

comparison_path = (
    FIGURE_DIR /
    "14_model_comparison_roc_auc.png"
)

plt.savefig(
    comparison_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(
    f"Saved visualization: {comparison_path}"
)


In [ ]:
# ============================================================
# ALL METRICS COMPARISON
# ============================================================

metrics_for_plot = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC"
]

comparison_long = results_df[
    ["Model"] + metrics_for_plot
].melt(
    id_vars="Model",
    var_name="Metric",
    value_name="Score"
)

plt.figure(
    figsize=(14, 7)
)

sns.barplot(
    data=comparison_long,
    x="Model",
    y="Score",
    hue="Metric",
    palette="viridis"
)

plt.title(
    "Machine Learning Model Performance Comparison",
    fontsize=18,
    fontweight="bold",
    color="#0B1F3A"
)

plt.xlabel("")
plt.ylabel("Score")
plt.ylim(0, 1)

plt.xticks(
    rotation=20,
    ha="right"
)

plt.legend(
    title="Metric",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()

all_metrics_path = (
    FIGURE_DIR /
    "15_model_comparison_all_metrics.png"
)

plt.savefig(
    all_metrics_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(
    f"Saved visualization: {all_metrics_path}"
)


## 7. Select the Best Model

The model with the highest mean cross-validated ROC-AUC is selected.

The selection criterion is based on cross-validation rather than
a single random train/test split.


In [ ]:
# ============================================================
# BEST MODEL
# ============================================================

best_model_name = results_df.iloc[0]["Model"]

best_model = models[best_model_name]

print(
    f"Selected model: {best_model_name}"
)

print(
    f"Cross-validated ROC-AUC: "
    f"{results_df.iloc[0]['ROC-AUC']:.4f}"
)


## 8. Build Final Pipeline

The selected model is combined with the preprocessing pipeline.

This ensures that the exact same transformations used during
training are automatically applied during prediction.


In [ ]:
# ============================================================
# FINAL PIPELINE
# ============================================================

final_pipeline = Pipeline(
    steps=[
        (
            "preprocessing",
            preprocessor
        ),
        (
            "model",
            best_model
        )
    ]
)

print(final_pipeline)


## 9. Holdout Evaluation

The model is evaluated on the 20% holdout set that was not used
during the model-comparison process.


In [ ]:
# ============================================================
# TRAIN FINAL SELECTED MODEL
# ============================================================

final_pipeline.fit(
    X_train,
    y_train
)

y_pred = final_pipeline.predict(
    X_holdout
)

y_probability = final_pipeline.predict_proba(
    X_holdout
)[:, 1]

print("Holdout predictions generated.")


In [ ]:
# ============================================================
# HOLDOUT METRICS
# ============================================================

holdout_metrics = {
    "Accuracy": accuracy_score(
        y_holdout,
        y_pred
    ),

    "Precision": precision_score(
        y_holdout,
        y_pred
    ),

    "Recall": recall_score(
        y_holdout,
        y_pred
    ),

    "F1": f1_score(
        y_holdout,
        y_pred
    ),

    "ROC-AUC": roc_auc_score(
        y_holdout,
        y_probability
    )
}

holdout_results = pd.DataFrame(
    holdout_metrics.items(),
    columns=[
        "Metric",
        "Score"
    ]
)

display(
    holdout_results.round(4)
)


## 10. Classification Report


In [ ]:
# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print(
    classification_report(
        y_holdout,
        y_pred,
        target_names=[
            "Did Not Survive",
            "Survived"
        ],
        digits=4
    )
)


## 11. Confusion Matrix


In [ ]:
# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_holdout,
    y_pred
)

plt.figure(
    figsize=(8, 6)
)

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    linewidths=1,
    linecolor="white",
    xticklabels=[
        "Predicted\nDid Not Survive",
        "Predicted\nSurvived"
    ],
    yticklabels=[
        "Actual\nDid Not Survive",
        "Actual\nSurvived"
    ]
)

plt.title(
    "Confusion Matrix — Titanic Survival",
    fontsize=18,
    fontweight="bold",
    color="#0B1F3A"
)

plt.xlabel("Prediction")
plt.ylabel("Actual")

plt.tight_layout()

cm_path = (
    FIGURE_DIR /
    "16_confusion_matrix.png"
)

plt.savefig(
    cm_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(
    f"Saved visualization: {cm_path}"
)


## 12. ROC Curve

The ROC curve evaluates how well the model distinguishes between
survivors and non-survivors across classification thresholds.


In [ ]:
# ============================================================
# ROC CURVE
# ============================================================

fig, ax = plt.subplots(
    figsize=(9, 7)
)

RocCurveDisplay.from_predictions(
    y_holdout,
    y_probability,
    ax=ax,
    color="#2563EB",
    name=best_model_name
)

ax.set_title(
    "ROC Curve — Titanic Survival",
    fontsize=18,
    fontweight="bold",
    color="#0B1F3A"
)

plt.tight_layout()

roc_path = (
    FIGURE_DIR /
    "17_roc_curve.png"
)

plt.savefig(
    roc_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(
    f"Saved visualization: {roc_path}"
)


## 13. Precision-Recall Curve


In [ ]:
# ============================================================
# PRECISION-RECALL CURVE
# ============================================================

fig, ax = plt.subplots(
    figsize=(9, 7)
)

PrecisionRecallDisplay.from_predictions(
    y_holdout,
    y_probability,
    ax=ax,
    color="#7C3AED",
    name=best_model_name
)

ax.set_title(
    "Precision-Recall Curve — Titanic Survival",
    fontsize=18,
    fontweight="bold",
    color="#0B1F3A"
)

plt.tight_layout()

pr_path = (
    FIGURE_DIR /
    "18_precision_recall_curve.png"
)

plt.savefig(
    pr_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(
    f"Saved visualization: {pr_path}"
)


## 14. Save Model

The complete preprocessing + model pipeline is saved as one
Joblib file.

This is important because the prediction application can later
load this single file and automatically perform the same
preprocessing used during training.


In [ ]:
# ============================================================
# SAVE TRAINED MODEL
# ============================================================

MODEL_DIR = (
    PROJECT_ROOT /
    "outputs" /
    "models"
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

model_path = (
    MODEL_DIR /
    "titanic_survival_model.joblib"
)

joblib.dump(
    final_pipeline,
    model_path
)

print(
    f"Model saved successfully:\n{model_path}"
)


In [ ]:
# ============================================================
# SAVE CROSS-VALIDATION RESULTS
# ============================================================

REPORT_DIR = (
    PROJECT_ROOT /
    "outputs" /
    "reports"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

cv_results_path = (
    REPORT_DIR /
    "model_comparison.csv"
)

results_df.to_csv(
    cv_results_path,
    index=False
)

holdout_results_path = (
    REPORT_DIR /
    "holdout_metrics.csv"
)

holdout_results.to_csv(
    holdout_results_path,
    index=False
)

print(
    f"Cross-validation results:\n{cv_results_path}"
)

print(
    f"\nHoldout results:\n{holdout_results_path}"
)


# Final Training Summary

The model-training stage is now complete.

### Outputs

- Cross-validation model comparison
- Accuracy
- Precision
- Recall
- F1 Score
- ROC-AUC
- Confusion matrix
- ROC curve
- Precision-recall curve
- Trained model pipeline

### Next Step

The next notebook, `05_model_interpretation.ipynb`, will analyze
which passenger characteristics are most strongly associated with
survival.

Special attention will be given to:

- Gender
- Passenger class / socio-economic status
- Age
- Family size
- Fare
- Cabin information
- Combined effects of multiple variables
